In [12]:
import sympy as sp
from sympy import cos, sin, pi, Integer, sqrt
import re

In [13]:
f = sp.Matrix([sp.Symbol(f"f[{i}]") for i in range(15)])
target = sp.Matrix([sp.Symbol(f"target[{i}]") for i in range(15)])
scales = sp.Matrix([sp.Symbol(f"scale_{i}") for i in ["x", "y", "z"]])
C = sp.Symbol("C")

alpha = sp.Symbol("alpha")
beta = sp.Symbol("beta")
gamma = sp.Symbol("gamma")

c, s, c2, c3, c4, s2, s3, s4 = sp.symbols("c,s,c2,c3,c4,s2,s3,s4")

In [14]:
# Matches decimal/scientific notation numbers but not integers
float_re = re.compile(
    r'(?<![\w.])'
    r'([+-]?(?:\d+\.\d*|\.\d+)(?:[eE][+-]?\d+)?'
    r'|[+-]?\d+[eE][+-]?\d+)'
)

def wrap_constants(code):
    return float_re.sub(r'constant<T>(\1)', code)

def to_ccode_vector(expr):
    ev = expr.evalf()
    out = "{\n"

    for i in range(len(ev)):
        code = sp.ccode(sp.simplify(ev[i]))
        code = wrap_constants(code)
        out += f"    {code}"
        if i != len(ev) - 1:
            out += ",\n"

    out += "\n};"
    return out

def to_ccode_matrix(expr):
    ev = sp.Matrix(expr).evalf()

    out = "{\n"

    for i in range(ev.rows):
        row = "    {"
        for j in range(ev.cols):
            code = sp.ccode(sp.simplify(ev[i, j]))
            code = wrap_constants(code)
            row += "" + code
            if j != ev.cols - 1:
                row += ","
            else:
                row += "}"
        out += row
        if i != ev.rows - 1:
            out += ",\n"
    out += "\n};"
    return out

def to_ccode_sym_matrix(expr):
    ev = sp.Matrix(expr).evalf()

    out = "{\n"

    for i in range(ev.rows):
        for j in range(i, ev.cols):
            code = sp.ccode(sp.simplify(ev[i, j]))
            code = wrap_constants(code)
            out += "    " + code
            if i != ev.rows - 1:
                out += ",\n"

    out += "};"
    return out

In [15]:
rotate_z_5d = sp.Matrix([
    [c2, 0, 0, 0, s2],
    [0, c, 0, s, 0],
    [0, 0, 1, 0, 0],
    [0, -s, 0, c, 0],
    [-s2, 0, 0, 0, c2]
])

rotate_z_9d = sp.Matrix([
    [c4, 0, 0, 0, 0, 0, 0, 0, s4],
    [0, c3, 0, 0, 0, 0, 0 , s3, 0],
    [0, 0, c2, 0, 0, 0, s2, 0, 0],
    [0, 0, 0, c, 0, s, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, -s, 0, c, 0, 0, 0],
    [0, 0, -s2, 0, 0, 0, c2, 0, 0],
    [0, -s3, 0, 0, 0, 0, 0, c3, 0],
    [-s4, 0, 0, 0, 0, 0, 0, 0, c4]
])

rotate_z = sp.diag(
    sp.Integer(1),
    rotate_z_5d,
    rotate_z_9d
)

rot_z_f = rotate_z * f

print(to_ccode_vector(rot_z_f))
#print(to_ccode_matrix(rotate_z))

{
    f[0],
    c2*f[1] + f[5]*s2,
    c*f[2] + f[4]*s,
    f[3],
    c*f[4] - f[2]*s,
    c2*f[5] - f[1]*s2,
    c4*f[6] + f[14]*s4,
    c3*f[7] + f[13]*s3,
    c2*f[8] + f[12]*s2,
    c*f[9] + f[11]*s,
    f[10],
    c*f[11] - f[9]*s,
    c2*f[12] - f[8]*s2,
    c3*f[13] - f[7]*s3,
    c4*f[14] - f[6]*s4
};


In [16]:
rot_x_pi_over_two_band_2 = sp.Matrix([
    [0, 0, 0, -1, 0],
    [0, -1, 0, 0, 0],
    [0, 0, -Integer(1)/2, 0, -sqrt(3)/2],
    [1, 0, 0, 0, 0],
    [0, 0, -sqrt(3)/2, 0, Integer(1)/2]
])

rot_x_pi_over_two_band_4 = sp.Matrix([
    [0, 0, 0, 0, 0, sqrt(14)/4, 0, - sqrt(2)/4, 0],
    [0, -Integer(3)/4, 0, sqrt(7)/4, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, sqrt(2)/4, 0, sqrt(14)/4, 0],
    [0, sqrt(7)/4, 0, Integer(3)/4, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, Integer(3)/8, 0, sqrt(5)/4, 0, sqrt(35)/8],
    [-sqrt(14)/4, 0, -sqrt(2)/4, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, sqrt(5)/4, 0, 1/2, 0, -sqrt(7)/4],
    [sqrt(2)/4, 0, -sqrt(14)/4, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, sqrt(35)/8, 0, -sqrt(7)/4, 0, Integer(1)/8]
])

rot_x_pi_2 = sp.diag(
    Integer(1),
    rot_x_pi_over_two_band_2,
    rot_x_pi_over_two_band_4
)


rotate_y = rot_x_pi_2 * rotate_z.T * rot_x_pi_2.T
rotate_y

rot_y_f = rotate_y * f
rot_y_f = sp.simplify(rot_y_f)

#print(to_ccode_matrix(rotate_y))
print(to_ccode_vector(rot_y_f))

{
    f[0],
    c*f[1] + f[2]*s,
    c*f[2] - f[1]*s,
    constant<T>(0.25)*f[3]*(constant<T>(3.0)*c2 + constant<T>(1.0)) - constant<T>(0.8660254037844386)*f[4]*s2 - constant<T>(0.4330127018922193)*f[5]*(c2 - constant<T>(1.0)),
    c2*f[4] + constant<T>(0.8660254037844386)*f[3]*s2 - constant<T>(0.5)*f[5]*s2,
    constant<T>(-0.4330127018922193)*f[3]*(c2 - constant<T>(1.0)) + constant<T>(0.5)*f[4]*s2 + constant<T>(0.25)*f[5]*(c2 + constant<T>(3.0)),
    constant<T>(0.125)*f[6]*(constant<T>(7.0)*c + c3) + constant<T>(0.088388347648318447)*f[7]*(constant<T>(7.0)*s + constant<T>(3.0)*s3) + constant<T>(0.33071891388307384)*f[8]*(c - c3) + constant<T>(0.23385358667337133)*f[9]*(constant<T>(3.0)*s - s3),
    constant<T>(-0.088388347648318447)*f[6]*(constant<T>(7.0)*s + constant<T>(3.0)*s3) + constant<T>(0.0625)*f[7]*(constant<T>(7.0)*c + constant<T>(9.0)*c3) - constant<T>(0.23385358667337133)*f[8]*(s - constant<T>(3.0)*s3) + constant<T>(0.49607837082461076)*f[9]*(c - c3),
    constant<T>(0.33

In [17]:
subs = {
    s: 1,
    c: 0,
    s2: 0,
    c2: -1,
    s3: -1,
    c3: 0,
    s4: 0,
    c4: 1,
}

rot_y_pi_over_two = rotate_y.subs(subs)

rotate_x = rot_y_pi_over_two.T * rotate_z.T * rot_y_pi_over_two

rot_x_f = rotate_x * f

#print(to_ccode_matrix(rotate_x))
print(to_ccode_vector(rot_x_f))

{
    f[0],
    c*f[1] - f[4]*s,
    c2*f[2] - constant<T>(0.8660254037844386)*f[3]*s2 - constant<T>(0.5)*f[5]*s2,
    constant<T>(0.8660254037844386)*f[2]*s2 + f[3]*(constant<T>(0.75)*c2 + constant<T>(0.25)) + constant<T>(0.4330127018922193)*f[5]*(c2 - 1),
    c*f[4] + f[1]*s,
    constant<T>(0.5)*f[2]*s2 + constant<T>(0.4330127018922193)*f[3]*(c2 - 1) + f[5]*(constant<T>(0.25)*c2 + constant<T>(0.75)),
    f[11]*(constant<T>(0.70156076002011403)*s - constant<T>(0.23385358667337133)*s3) - f[13]*(constant<T>(0.61871843353822908)*s + constant<T>(0.26516504294495535)*s3) + f[6]*(constant<T>(0.875)*c + constant<T>(0.125)*c3) - constant<T>(0.33071891388307384)*f[8]*(c - c3),
    f[10]*(constant<T>(0.52291251658379723)*s2 - constant<T>(0.26145625829189861)*s4) - f[12]*(constant<T>(0.46770717334674267)*s2 + constant<T>(0.23385358667337133)*s4) - f[14]*(constant<T>(0.61871843353822908)*s2 + constant<T>(0.044194173824159223)*s4) + f[7]*(constant<T>(0.875)*c2 + constant<T>(0.125)*c4) - constant<

# Derivatives

In [18]:

L_x_5d = sp.Matrix([
    [0, 0, 0, -1, 0],
    [0, 0, -sqrt(3), 0, -1],
    [0, sqrt(3), 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0]
])

L_y_5d = sp.Matrix([
    [0, 1, 0, 0, 0],
    [-1, 0, 0, 0, 0],
    [0, 0, 0, -sqrt(3), 0],
    [0, 0, sqrt(3), 0, -1],
    [0, 0, 0, 1, 0]
])

L_z_5d = sp.Matrix([
    [0, 0, 0, 0, 2],
    [0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0],
    [0, -1, 0, 0, 0],
    [-2, 0, 0, 0, 0]
])

L_x_9d = sp.Matrix([
    [0, 0, 0, 0, 0, 0, 0, -sqrt(2), 0],
    [0, 0, 0, 0, 0, 0, -sqrt(Integer(7)/2), 0, -sqrt(2)],
    [0, 0, 0, 0, 0, -Integer(3)/sqrt(2), 0, -sqrt(Integer(7)/2), 0],
    [0, 0, 0, 0, -sqrt(10), 0, -Integer(3)/sqrt(2), 0, 0],
    [0, 0, 0, sqrt(10), 0, 0, 0, 0, 0],
    [0, 0, 3/sqrt(2), 0, 0, 0, 0, 0, 0],
    [0, sqrt(Integer(7)/2), 0, Integer(3)/sqrt(2), 0, 0, 0, 0, 0],
    [sqrt(2), 0, sqrt(Integer(7)/2), 0, 0, 0, 0, 0, 0],
    [0, sqrt(2), 0, 0, 0, 0, 0, 0, 0]
])

L_y_9d = sp.Matrix([
    [0, sqrt(2), 0, 0, 0, 0, 0, 0, 0],
    [-sqrt(2), 0, sqrt(Integer(7)/2), 0, 0, 0, 0, 0, 0],
    [0, -sqrt(Integer(7)/2), 0, Integer(3)/sqrt(2), 0, 0, 0, 0, 0],
    [0, 0, -Integer(3)/sqrt(2), 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, -sqrt(10), 0, 0, 0],
    [0, 0, 0, 0, sqrt(10), 0, -3/sqrt(2), 0, 0],
    [0, 0, 0, 0, 0, 3/sqrt(2), 0, -sqrt(Integer(7)/2), 0],
    [0, 0, 0, 0, 0, 0, sqrt(Integer(7)/2), 0, -sqrt(2)],
    [0, 0, 0, 0, 0, 0, 0, sqrt(2), 0]
])

L_z_9d = sp.Matrix([
    [0, 0, 0, 0, 0, 0, 0, 0, 4],
    [0, 0, 0, 0, 0, 0, 0, 3, 0],
    [0, 0, 0, 0, 0, 0, 2, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, -1, 0, 0, 0, 0, 0],
    [0, 0, -2, 0, 0, 0, 0, 0, 0],
    [0, -3, 0, 0, 0, 0, 0, 0, 0],
    [-4, 0, 0, 0, 0, 0, 0, 0, 0]
])

L_x = sp.diag(
    Integer(0),
    L_x_5d,
    L_x_9d
)

L_y = sp.diag(
    Integer(0),
    L_y_5d,
    L_y_9d
)

L_z = sp.diag(
    Integer(0),
    L_z_5d,
    L_z_9d
)

F = sp.Matrix([
    [sqrt(pi) * Integer(2) / 5, sqrt(pi) * Integer(2) / 5, sqrt(pi) * Integer(2) / 5],

    [0, 0, 0],
    [0, 0, 0],
    [-(Integer(4) * sqrt(pi)) / (7 * sqrt(5)), -(4 * sqrt(pi)) / (7 * sqrt(5)), 2 * (4 * sqrt(pi)) / (7 * sqrt(5))],
    [0, 0, 0],
    [(4 * sqrt(3 * pi)) / (7 * sqrt(5)), -(4 * sqrt(3 * pi)) / (7 * sqrt(5)), 0],

    [0, 0, 0],
    [0, 0, 0],
    [0, 0, 0],
    [0, 0, 0],
    [2 * sqrt(pi) / 35, 2 * sqrt(pi) / 35, 16 * sqrt(pi) / 105],
    [0, 0, 0],
    [-(4 * sqrt(pi)) / (21 * sqrt(5)), (4 * sqrt(pi)) / (21 * sqrt(5)), 0],
    [0, 0, 0],
    [(2 * sqrt(pi)) / (3 * sqrt(35)), (2 * sqrt(pi)) / (3 * sqrt(35)), 0]
])


grad = sp.Matrix.vstack(
    sp.Matrix([
        -2 * target.T * L_x * F * scales,
        -2 * target.T * L_y * F * scales,
        -2 * target.T * L_z * F * scales
    ]),
    2 * F.T * F * scales - 2 * F.T * target
)

grad[3] -= C / scales[0]
grad[4] -= C / scales[1]
grad[5] -= C / scales[2]

print(to_ccode_vector(grad))

{
    constant<T>(1.1102230246251565e-16)*scale_x*target[7] + scale_y*(constant<T>(-3.1381413698186362)*target[2] + constant<T>(1.1298600273745016)*target[7] + constant<T>(1.2811408494623837)*target[9]) + scale_z*(constant<T>(3.1381413698186362)*target[2] + constant<T>(1.7081877992831784)*target[9]),
    scale_x*(constant<T>(-1.2811408494623837)*target[11] + constant<T>(1.1298600273745016)*target[13] + constant<T>(3.1381413698186362)*target[4]) + constant<T>(1.1102230246251565e-16)*scale_y*target[13] - scale_z*(constant<T>(1.7081877992831784)*target[11] + constant<T>(3.1381413698186362)*target[4]),
    -scale_x*(constant<T>(3.1381413698186362)*target[1] + constant<T>(1.5978633742962567)*target[6] - constant<T>(0.60393558820663018)*target[8]) - scale_y*(constant<T>(-3.1381413698186362)*target[1] + constant<T>(1.5978633742962567)*target[6] + constant<T>(0.60393558820663018)*target[8]),
    -C/scale_x + constant<T>(2.7925268031909272)*scale_x + constant<T>(0.23935944027350806)*scale_y + c

In [19]:
# reference frame

ref_frame = F * scales

print(to_ccode_vector(ref_frame))

{
    constant<T>(0.7089815403622064)*scale_x + constant<T>(0.7089815403622064)*scale_y + constant<T>(0.7089815403622064)*scale_z,
    0,
    0,
    constant<T>(-0.45295169115497264)*scale_x - constant<T>(0.45295169115497264)*scale_y + constant<T>(0.90590338230994527)*scale_z,
    0,
    constant<T>(0.78453534245465906)*scale_x - constant<T>(0.78453534245465906)*scale_y,
    0,
    0,
    0,
    0,
    constant<T>(0.1012830771946009)*scale_x + constant<T>(0.1012830771946009)*scale_y + constant<T>(0.27008820585226911)*scale_z,
    0,
    constant<T>(-0.15098389705165755)*scale_x + constant<T>(0.15098389705165755)*scale_y,
    0,
    constant<T>(0.19973292178703209)*scale_x + constant<T>(0.19973292178703209)*scale_y
};


In [20]:
h1 = -2 * sp.Matrix([
    [target.T * L_x * L_x * F * scales, target.T * L_x * L_y * F * scales, target.T * L_x * L_z * F * scales],
    [target.T * L_y * L_x * F * scales, target.T * L_y * L_y * F * scales, target.T * L_y * L_z * F * scales],
    [target.T * L_z * L_x * F * scales, target.T * L_z * L_y * F * scales, target.T * L_z * L_z * F * scales]
])

h2 = -2 * sp.Matrix([
    (F.T * L_x.T * target).T,
    (F.T * L_y.T * target).T,
    (F.T * L_z.T * target).T
])

h3 = h2.T

h4 = 2 * F.T * F + C * sp.Matrix([[1 / (scales[0]**2), 0, 0], [0, 1/ (scales[1]**2), 0], [0, 0, 1 / (scales[2] ** 2)]])

hess = sp.BlockMatrix([
    [h1, h2],
    [h3, h4]
])

print(to_ccode_matrix(sp.Matrix(hess)))

{
    {constant<T>(2.0)*scale_x*(constant<T>(2.2204460492503131e-16)*target[12] + constant<T>(5.5511151231257827e-17)*target[14]) + constant<T>(2.0)*scale_y*(constant<T>(2.0256615438920185)*target[10] + constant<T>(2.4157423528265207)*target[12] + constant<T>(0.79893168714812823)*target[14] - constant<T>(2.7177101469298357)*target[3] - constant<T>(1.5690706849093181)*target[5]) + constant<T>(2.0)*scale_z*(constant<T>(2.7008820585226916)*target[10] + constant<T>(1.8118067646198912)*target[12] + constant<T>(2.7177101469298357)*target[3] + constant<T>(1.5690706849093181)*target[5]),constant<T>(-2.0)*scale_x*(constant<T>(1.5690706849093181)*target[1] + constant<T>(0.79893168714812823)*target[6] - constant<T>(0.3019677941033152)*target[8]) - constant<T>(1.1102230246251565e-16)*scale_y*(target[6] - target[8]) + constant<T>(2.0)*scale_z*(constant<T>(1.5690706849093181)*target[1] + constant<T>(1.8118067646198905)*target[8]),constant<T>(-2.0)*scale_x*(constant<T>(-0.64057042473119186)*target[11

# Z aligned

In [21]:
# gradient

scales = sp.Matrix([sp.Symbol("scale_x"), sp.Symbol("scale_y"), Integer(1)])

grad = sp.Matrix([
    -2 * target.T * L_z * F * scales
])

grad_scales_full = 2 * F.T * F * scales - 2 * F.T * target

grad = sp.Matrix([
    grad,
    grad_scales_full[0] - C * scales[0],
    grad_scales_full[1] - C * scales[1]
])

print(to_ccode_vector(grad))

{
    -scale_x*(constant<T>(3.1381413698186362)*target[1] + constant<T>(1.5978633742962567)*target[6] - constant<T>(0.60393558820663018)*target[8]) - scale_y*(constant<T>(-3.1381413698186362)*target[1] + constant<T>(1.5978633742962567)*target[6] + constant<T>(0.60393558820663018)*target[8]),
    -C*scale_x + constant<T>(2.7925268031909272)*scale_x + constant<T>(0.23935944027350806)*scale_y - constant<T>(1.4179630807244128)*target[0] - constant<T>(0.20256615438920181)*target[10] + constant<T>(0.30196779410331509)*target[12] - constant<T>(0.39946584357406417)*target[14] + constant<T>(0.90590338230994527)*target[3] - constant<T>(1.5690706849093181)*target[5] + constant<T>(0.23935944027350806),
    -C*scale_y + constant<T>(0.23935944027350806)*scale_x + constant<T>(2.7925268031909272)*scale_y - constant<T>(1.4179630807244128)*target[0] - constant<T>(0.20256615438920181)*target[10] - constant<T>(0.30196779410331509)*target[12] - constant<T>(0.39946584357406417)*target[14] + constant<T>(0.90

In [22]:
# hessian

h00 = -2 * target.T * L_z * L_z * F * scales
h00 = h00[0, 0]

hess_theta_full = -2 * F.T * L_z.T * target
hess_lam_full = 2 * F.T * F

hess = sp.Matrix([
    [h00, hess_theta_full[0], hess_theta_full[1]],
    [hess_theta_full[0], hess_lam_full[0, 0] + C / (scales[0] ** 2), hess_lam_full[0, 1]],
    [hess_theta_full[1], hess_lam_full[1, 0], hess_lam_full[1, 1] + C / (scales[1] ** 2)]
])

print(to_ccode_matrix(hess))

{
    {scale_x*(constant<T>(-1.2078711764132604)*target[12] + constant<T>(6.3914534971850268)*target[14] + constant<T>(6.2762827396372725)*target[5]) + scale_y*(constant<T>(1.2078711764132604)*target[12] + constant<T>(6.3914534971850268)*target[14] - constant<T>(6.2762827396372725)*target[5]),constant<T>(-3.1381413698186362)*target[1] - constant<T>(1.5978633742962567)*target[6] + constant<T>(0.60393558820663018)*target[8],constant<T>(3.1381413698186362)*target[1] - constant<T>(1.5978633742962567)*target[6] - constant<T>(0.60393558820663018)*target[8]},
    {constant<T>(-3.1381413698186362)*target[1] - constant<T>(1.5978633742962567)*target[6] + constant<T>(0.60393558820663018)*target[8],C/pow(scale_x, 2) + constant<T>(2.7925268031909272),constant<T>(0.23935944027350806)},
    {constant<T>(3.1381413698186362)*target[1] - constant<T>(1.5978633742962567)*target[6] - constant<T>(0.60393558820663018)*target[8],constant<T>(0.23935944027350806),C/pow(scale_y, 2) + constant<T>(2.79252680319092